**Pipeline de Tratamento e Treinamento da Base de Dados**

A base utilizada encontra-se em https://data.mendeley.com/datasets/yvb7khhbrj1

In [5]:
"""
Pipeline de Machine Learning — Detecção de Adulteração de Combustível
Dataset: Mendeley — etanol hidratado com diferentes frações de água
Label: binário — puro (0% água) vs adulterado (10–50% água)
Balanceamento: class_weight='balanced'
Validação: split 80/20 estratificado com 10 seeds diferentes
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              confusion_matrix, roc_curve,
                              precision_score, recall_score)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =============================================================================
# CARREGAMENTO E CRIAÇÃO DO LABEL
# =============================================================================
df = pd.read_excel('supplementary-data.xlsx', sheet_name='Raw')

# Label binário: 0 = puro (0% água), 1 = adulterado (>=10% água)
df['label'] = (df['Water fraction [0-1]'] > 0.0).astype(int)

features = [
    'Rotational speed (rpm)',
    'Compression ratio',
    'Ignition Advance Angle (°BTDC)',
    'Fuel temperature (℃)',
    'Exhaust temperature (℃)',
    'Oil temperature (℃)',
    'Torque (N m)',
    'Fuel consumption (g/s)',
    'Power (kW)',
    'Specific fuel consumption (g/kWh)',
    'Efficiency (%)',
    'CO (%)',
    'CO2 (%)',
    'O2 (%)',
    'HC (ppm)',
    'NOx (ppm)'
]

features_abrev = [
    'RPM', 'Comp. ratio', 'Ign. advance', 'Fuel temp.',
    'Exhaust temp.', 'Oil temp.', 'Torque', 'Fuel consump.',
    'Power', 'BSFC', 'Efficiency', 'CO', 'CO2', 'O2', 'HC', 'NOx'
]

X = df[features].values
y = df['label'].values

print("=" * 60)
print("DISTRIBUIÇÃO DO LABEL")
print("=" * 60)
print(f"Puro (0):       {(y == 0).sum()} amostras")
print(f"Adulterado (1): {(y == 1).sum()} amostras")
print(f"Razão:          1:{(y==1).sum() // (y==0).sum()}")

DISTRIBUIÇÃO DO LABEL
Puro (0):       60 amostras
Adulterado (1): 300 amostras
Razão:          1:5


In [10]:
# =============================================================================
# MODELOS com class_weight='balanced', XGBoost usa scale_pos_weight como equivalente
# =============================================================================
scale_pos = (y == 0).sum() / (y == 1).sum()  # para XGBoost

modelos = {
    'KNN (k=5)':     KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF)':     SVC(kernel='rbf', class_weight='balanced',
                         probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200,
                                             class_weight='balanced',
                                             random_state=42),
    'XGBoost':       XGBClassifier(n_estimators=200,
                                    scale_pos_weight=1/scale_pos,
                                    random_state=42,
                                    eval_metric='logloss',
                                    verbosity=0),
}

In [11]:
# =============================================================================
# EXPERIMENTO: 10 SEEDS — SPLIT 80/20 ESTRATIFICADO
# =============================================================================
seeds = [0, 1, 2, 7, 13, 21, 42, 99, 123, 777]

metricas_por_modelo = {
    nome: {'acc': [], 'f1_macro': [], 'f1_puro': [], 'f1_adult': [],
           'prec_macro': [], 'rec_macro': [],
           'prec_puro': [], 'rec_puro': [],
           'prec_adult': [], 'rec_adult': [], 'auc': []}
    for nome in modelos
}
ultimo_resultado = {}

for seed in seeds:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    for nome, modelo in modelos.items():
        m = type(modelo)(**modelo.get_params())
        m.fit(X_train_sc, y_train)
        y_pred = m.predict(X_test_sc)
        y_prob = m.predict_proba(X_test_sc)[:, 1]

        metricas_por_modelo[nome]['acc'].append(accuracy_score(y_test, y_pred))
        metricas_por_modelo[nome]['f1_macro'].append(f1_score(y_test, y_pred, average='macro'))
        metricas_por_modelo[nome]['f1_puro'].append(f1_score(y_test, y_pred, pos_label=0, average='binary'))
        metricas_por_modelo[nome]['f1_adult'].append(f1_score(y_test, y_pred, pos_label=1, average='binary'))
        metricas_por_modelo[nome]['prec_macro'].append(precision_score(y_test, y_pred, average='macro', zero_division=0))
        metricas_por_modelo[nome]['rec_macro'].append(recall_score(y_test, y_pred, average='macro', zero_division=0))
        metricas_por_modelo[nome]['prec_puro'].append(precision_score(y_test, y_pred, pos_label=0, average='binary', zero_division=0))
        metricas_por_modelo[nome]['rec_puro'].append(recall_score(y_test, y_pred, pos_label=0, average='binary', zero_division=0))
        metricas_por_modelo[nome]['prec_adult'].append(precision_score(y_test, y_pred, pos_label=1, average='binary', zero_division=0))
        metricas_por_modelo[nome]['rec_adult'].append(recall_score(y_test, y_pred, pos_label=1, average='binary', zero_division=0))
        metricas_por_modelo[nome]['auc'].append(roc_auc_score(y_test, y_prob))

        if seed == 42:
            ultimo_resultado[nome] = {
                'y_test': y_test, 'y_pred': y_pred,
                'y_prob': y_prob, 'modelo': m
            }

In [12]:
# =============================================================================
# TABELA DE RESULTADOS
# =============================================================================
nomes = list(modelos.keys())
resumo = {nome: {k: np.array(v) for k, v in metricas_por_modelo[nome].items()}
          for nome in nomes}

print("\n" + "=" * 78)
print("RESULTADOS — média ± desvio padrão (10 seeds, class_weight=balanced)")
print("=" * 78)

for classe, sufixo in [('MACRO (geral)', 'macro'), ('PURO', 'puro'), ('ADULTERADO', 'adult')]:
    print(f"\n--- {classe} ---")
    print(f"{'Modelo':<18} {'Precisão':>12} {'Recall':>12} {'F1':>12} {'Acurácia':>12} {'AUC':>10}")
    print('-' * 70)
    for nome in nomes:
        m = resumo[nome]
        p   = m[f'prec_{sufixo}']
        r   = m[f'rec_{sufixo}']
        f   = m[f'f1_{sufixo}']
        acc = m['acc']
        auc = m['auc']
        print(f"{nome:<18} {p.mean():.3f}±{p.std():.3f}  {r.mean():.3f}±{r.std():.3f}  "
              f"{f.mean():.3f}±{f.std():.3f}  {acc.mean():.3f}±{acc.std():.3f}  "
              f"{auc.mean():.3f}±{auc.std():.3f}")



RESULTADOS — média ± desvio padrão (10 seeds, class_weight=balanced)

--- MACRO (geral) ---
Modelo                 Precisão       Recall           F1     Acurácia        AUC
----------------------------------------------------------------------
KNN (k=5)          1.000±0.000  1.000±0.000  1.000±0.000  1.000±0.000  1.000±0.000
SVM (RBF)          1.000±0.000  1.000±0.000  1.000±0.000  1.000±0.000  1.000±0.000
Random Forest      0.995±0.005  0.975±0.028  0.984±0.018  0.992±0.009  1.000±0.000
XGBoost            0.996±0.004  0.979±0.021  0.987±0.013  0.993±0.007  1.000±0.000

--- PURO ---
Modelo                 Precisão       Recall           F1     Acurácia        AUC
----------------------------------------------------------------------
KNN (k=5)          1.000±0.000  1.000±0.000  1.000±0.000  1.000±0.000  1.000±0.000
SVM (RBF)          1.000±0.000  1.000±0.000  1.000±0.000  1.000±0.000  1.000±0.000
Random Forest      1.000±0.000  0.950±0.055  0.974±0.030  0.992±0.009  1.000±0.000
XGBoos

In [13]:
# =============================================================================
# FEATURE IMPORTANCE (Random Forest — seed=42)
# =============================================================================
rf_modelo = ultimo_resultado['Random Forest']['modelo']
importances = pd.Series(rf_modelo.feature_importances_,
                        index=features_abrev).sort_values(ascending=False)

print("\n" + "=" * 60)
print("FEATURE IMPORTANCE — Random Forest (seed=42)")
print("=" * 60)
for feat, imp in importances.items():
    bar = '█' * int(imp * 50)
    print(f"  {feat:<25} {imp:.4f}  {bar}")



FEATURE IMPORTANCE — Random Forest (seed=42)
  BSFC                      0.2748  █████████████
  Oil temp.                 0.2612  █████████████
  NOx                       0.1353  ██████
  Fuel consump.             0.0744  ███
  Fuel temp.                0.0535  ██
  Efficiency                0.0495  ██
  Torque                    0.0418  ██
  HC                        0.0345  █
  Ign. advance              0.0269  █
  O2                        0.0217  █
  CO                        0.0071  
  CO2                       0.0065  
  Power                     0.0049  
  Exhaust temp.             0.0041  
  Comp. ratio               0.0023  
  RPM                       0.0014  


In [14]:
# =============================================================================
# FIGURAS
# =============================================================================
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'figure.dpi': 150
})

cores_mod = ['#E53935', '#1565C0', '#2E7D32', '#F57F17']

# --- Figura 1: Comparativo de modelos ---
metricas_keys  = ['acc', 'f1_macro', 'f1_puro', 'auc']
metricas_nomes = ['Acurácia', 'F1-macro', 'F1-puro', 'AUC-ROC']
cores_bar = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(nomes))
width = 0.18
for j, (mk, mn, cor) in enumerate(zip(metricas_keys, metricas_nomes, cores_bar)):
    vals = [resumo[n][mk].mean() for n in nomes]
    errs = [resumo[n][mk].std()  for n in nomes]
    ax.bar(x + j * width, vals, width, label=mn, color=cor,
           alpha=0.85, yerr=errs, capsize=3, error_kw={'linewidth': 1.2})

ax.set_title('Comparativo de modelos — Detecção de adulteração\n'
             '(média ± desvio padrão, 10 seeds, class_weight=balanced, split 80/20)',
             fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(nomes, rotation=15, ha='right')
ax.set_ylim(0.5, 1.08)
ax.set_ylabel('Score')
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig1_comparativo_modelos.png', bbox_inches='tight', dpi=150)
plt.close()
print("\n[Figura salva] fig1_comparativo_modelos.png")

# --- Figura 2: Feature Importance ---
fig, ax = plt.subplots(figsize=(9, 6))
cores_imp = ['#1565C0' if i < 3 else '#42A5F5' if i < 6
             else '#BBDEFB' for i in range(len(importances))]
ax.barh(importances.index[::-1], importances.values[::-1],
        color=cores_imp[::-1], edgecolor='white', linewidth=0.5)
for i, val in enumerate(importances.values[::-1]):
    ax.text(val + 0.003, i, f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Importância relativa (Gini)')
ax.set_title('Importância de features — Random Forest\n'
             '(seed=42, class_weight=balanced)', fontweight='bold')
ax.set_xlim(0, importances.max() * 1.2)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.savefig('fig2_feature_importance.png', bbox_inches='tight', dpi=150)
plt.close()
print("[Figura salva] fig2_feature_importance.png")

# --- Figura 3: Confusion matrices (seed=42) ---
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
fig.suptitle('Matrizes de confusão — seed=42, class_weight=balanced, split 80/20',
             fontweight='bold')
for ax, nome in zip(axes, nomes):
    cm = confusion_matrix(ultimo_resultado[nome]['y_test'],
                          ultimo_resultado[nome]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Puro', 'Adult.'],
                yticklabels=['Puro', 'Adult.'],
                cbar=False, linewidths=0.5)
    acc = accuracy_score(ultimo_resultado[nome]['y_test'],
                         ultimo_resultado[nome]['y_pred'])
    ax.set_title(f'{nome}\nacc={acc:.3f}', fontsize=10)
    ax.set_xlabel('Predito', fontsize=9)
    ax.set_ylabel('Real', fontsize=9)
plt.tight_layout()
plt.savefig('fig3_confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.close()
print("[Figura salva] fig3_confusion_matrices.png")

# --- Figura 4: Curvas ROC (seed=42) ---
fig, ax = plt.subplots(figsize=(7, 6))
cores_roc = ['#E53935', '#1565C0', '#2E7D32', '#F57F17']
for nome, cor in zip(nomes, cores_roc):
    fpr, tpr, _ = roc_curve(ultimo_resultado[nome]['y_test'],
                             ultimo_resultado[nome]['y_prob'])
    auc_val = roc_auc_score(ultimo_resultado[nome]['y_test'],
                             ultimo_resultado[nome]['y_prob'])
    ax.plot(fpr, tpr, lw=2, color=cor,
            label=f'{nome} (AUC={auc_val:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4, label='Aleatório')
ax.set_xlabel('Taxa de Falso Positivo')
ax.set_ylabel('Taxa de Verdadeiro Positivo')
ax.set_title('Curvas ROC — seed=42, class_weight=balanced, split 80/20',
             fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('fig4_curvas_roc.png', bbox_inches='tight', dpi=150)
plt.close()
print("[Figura salva] fig4_curvas_roc.png")

# --- Figura 5: Distribuição das 6 features mais importantes ---
top6_idx   = np.argsort(rf_modelo.feature_importances_)[::-1][:6]
top6_feats = [features[i] for i in top6_idx]
top6_abrev = [features_abrev[i] for i in top6_idx]

df['Classe'] = df['label'].map({0: 'Puro', 1: 'Adulterado'})
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
fig.suptitle('Distribuição das 6 features mais importantes por classe',
             fontweight='bold')
palette = {'Puro': '#1565C0', 'Adulterado': '#E53935'}
for ax, feat, abrev in zip(axes.flat, top6_feats, top6_abrev):
    sns.boxplot(data=df, x='Classe', y=feat, ax=ax,
                palette=palette, width=0.5,
                order=['Puro', 'Adulterado'])
    ax.set_title(abrev, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig5_distribuicao_features.png', bbox_inches='tight', dpi=150)
plt.close()
print("[Figura salva] fig5_distribuicao_features.png")

# --- Figura 6: Precisão, Recall e F1 por modelo (macro) ---
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Precisão, Recall e F1 — macro (10 seeds, class_weight=balanced)',
             fontweight='bold')
for ax, mk, titulo in zip(axes,
                           ['prec_macro', 'rec_macro', 'f1_macro'],
                           ['Precisão (macro)', 'Recall (macro)', 'F1 (macro)']):
    vals = [resumo[n][mk].mean() for n in nomes]
    errs = [resumo[n][mk].std()  for n in nomes]
    bars = ax.bar(nomes, vals, color=cores_mod, alpha=0.85,
                  yerr=errs, capsize=4, error_kw={'linewidth': 1.2})
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylim(0.5, 1.12)
    ax.set_xticklabels(nomes, rotation=20, ha='right', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig6_precisao_recall_f1.png', bbox_inches='tight', dpi=150)
plt.close()
print("[Figura salva] fig6_precisao_recall_f1.png")

# --- Figura 7: Precisão vs Recall por classe ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Precisão vs Recall por classe (10 seeds, class_weight=balanced)',
             fontweight='bold')
for ax, sufixo, titulo in zip(axes,
                               ['puro', 'adult'],
                               ['Classe: Puro', 'Classe: Adulterado']):
    prec_vals = [resumo[n][f'prec_{sufixo}'].mean() for n in nomes]
    rec_vals  = [resumo[n][f'rec_{sufixo}'].mean()  for n in nomes]
    prec_errs = [resumo[n][f'prec_{sufixo}'].std()  for n in nomes]
    rec_errs  = [resumo[n][f'rec_{sufixo}'].std()   for n in nomes]

    x = np.arange(len(nomes))
    width = 0.35
    ax.bar(x - width/2, prec_vals, width, label='Precisão',
           color='#1565C0', alpha=0.85, yerr=prec_errs, capsize=3)
    ax.bar(x + width/2, rec_vals,  width, label='Recall',
           color='#E53935', alpha=0.85, yerr=rec_errs,  capsize=3)
    for i, (p, r) in enumerate(zip(prec_vals, rec_vals)):
        ax.text(i - width/2, p + 0.005, f'{p:.3f}', ha='center', va='bottom', fontsize=8)
        ax.text(i + width/2, r + 0.005, f'{r:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_title(titulo, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(nomes, rotation=20, ha='right', fontsize=9)
    ax.set_ylim(0.5, 1.15)
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig7_precisao_recall_classes.png', bbox_inches='tight', dpi=150)
plt.close()
print("[Figura salva] fig7_precisao_recall_classes.png")


[Figura salva] fig1_comparativo_modelos.png
[Figura salva] fig2_feature_importance.png
[Figura salva] fig3_confusion_matrices.png
[Figura salva] fig4_curvas_roc.png
[Figura salva] fig5_distribuicao_features.png
[Figura salva] fig6_precisao_recall_f1.png
[Figura salva] fig7_precisao_recall_classes.png
